# 1. Настройка окружения

**Цель:** Проверить установку PyTorch, определить доступные устройства (CPU/GPU/MPS), настроить логирование.

---

## 🎭 Мем дня

*Проверка окружения: то самое чувство, когда код работает только на твоей машине, а у коллег — «CUDA not available»…

Но ничего, MPS тоже справляется!*

![Мем про окружение](memes/01_env.png)


In [1]:
# Импортируем стандартные библиотеки Python
import sys
import os

# Вместо logging используем простой print — учебный курс, зачем нам логи?
import builtins as _builtins
def _log(msg, *args):
    if args:
        _builtins.print(msg % args)
    else:
        _builtins.print(msg)

import types
log = types.SimpleNamespace(info=_log, debug=_log, warning=_log, error=_log)







In [2]:
# Импортируем PyTorch — главный фреймворк для всех наших экспериментов
import torch

log.debug("Импортируем PyTorch")
print(f"PyTorch version: {torch.__version__}")
log.info("PyTorch %s загружен", torch.__version__)


Импортируем PyTorch
PyTorch version: 2.8.0
PyTorch 2.8.0 загружен


In [3]:
# Информация о системе: версия Python и платформа
import platform

print(f"Python: {sys.version}")           # версия интерпретатора
print(f"Platform: {platform.platform()}") # ОС и архитектура
print(f"Processor: {platform.processor()}")  # процессор
log.debug("Информация о системе получена")


Python: 3.9.6 (default, Apr 17 2026, 18:15:52) 
[Clang 21.0.0 (clang-2100.1.1.101)]
Platform: macOS-26.2-arm64-arm-64bit
Processor: arm
Информация о системе получена


In [4]:
# === Проверка CUDA (NVIDIA GPU) ===
# CUDA позволяет запускать тензорные операции на видеокартах NVIDIA.
# torch.cuda.is_available() — проверяет, есть ли доступная CUDA-карта.
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    # Если CUDA есть — покажем имя GPU и версию CUDA toolkit
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    log.info("CUDA устройство: %s", torch.cuda.get_device_name(0))
else:
    log.warning("CUDA не доступен — either no NVIDIA GPU or no CUDA toolkit")


CUDA available: False
CUDA не доступен — either no NVIDIA GPU or no CUDA toolkit


In [5]:
# Проверка MPS (Apple Silicon)
mps_available = torch.backends.mps.is_available()
mps_built = torch.backends.mps.is_built()
print(f"MPS available: {mps_available}")
print(f"MPS built: {mps_built}")
if mps_available:
    log.info("MPS доступен — will use Apple Silicon GPU")
else:
    log.warning("MPS не доступен")

MPS available: True
MPS built: True
MPS доступен — will use Apple Silicon GPU


## Почему важен выбор устройства (device)?

PyTorch поддерживает три типа устройств для вычислений:

| Устройство | Когда доступно | Скорость | Применение |
|------------|----------------|----------|------------|
| **CPU** | Всегда | Медленно | Отладка, маленькие модели |
| **CUDA** | NVIDIA GPU | ×10–100 быстрее CPU | Большие модели, batch-обработка |
| **MPS** | Apple Silicon (M1+) | ×5–20 быстрее CPU | То же, на Mac |

**Почему это важно:**
- Обучение трансформеров даже на маленьких датасетах занимает минуты на CPU и секунды на GPU
- В этом курсе мы будем использовать MPS (Apple Silicon) или CPU, если MPS недоступен
- Все тензоры нужно явно перемещать на устройство: `tensor.to(device)`
- Передача данных между CPU и GPU — дорогая операция (overhead), старайтесь держать данные на одном устройстве

Подробнее: [PyTorch MPS Documentation](https://pytorch.org/docs/stable/notes/mps.html)

In [6]:
# === Выбор устройства ===
# Приоритет: CUDA > MPS > CPU
# Это стандартный паттерн для всех PyTorch-проектов.
if torch.cuda.is_available():
    device = torch.device("cuda")       # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")        # Apple Silicon GPU
else:
    device = torch.device("cpu")        # Универсальное падение

print(f"Selected device: {device}")
log.info("Используем устройство: %s", device)


Selected device: mps
Используем устройство: mps


In [7]:
# === Тестовый тензор ===
# torch.randn(3, 3) создаёт матрицу 3×3 из случайных чисел
# Параметр device=device перемещает тензор на выбранное устройство сразу при создании
# Это эффективнее, чем создать на CPU и потом .to(device)
x = torch.randn(3, 3, device=device)
print(f"Test tensor shape: {x.shape}")    # размерность (rows, cols)
print(f"Test tensor device: {x.device}")  # на каком устройстве лежит
print(f"Test tensor dtype: {x.dtype}")    # тип данных (float32 по умолчанию)
print(x)
log.debug("Тестовый тензор создан на %s: форма=%s", device, x.shape)


Test tensor shape: torch.Size([3, 3])
Test tensor device: mps:0
Test tensor dtype: torch.float32


tensor([[-0.2834,  0.2422,  1.2105],
        [-0.6868, -0.0330, -0.5975],
        [ 0.3706, -0.1039, -0.0028]], device='mps:0')
Тестовый тензор создан на mps: форма=torch.Size([3, 3])


In [8]:
# === Базовый benchmark: умножение матриц ===
# Это простой тест производительности вычислительного устройства.
# Матричное умножение — основа attention (Q @ K.T), поэтому скорость важна.
import time

sizes = [100, 1000, 5000]  # размеры матриц: от маленьких до больших
for n in sizes:
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    
    # Прогрев: первые запуски могут быть медленнее (JIT, выделение памяти)
    for _ in range(5):
        c = a @ b
    
    # На GPU операции асинхронные — нужно явно дождаться завершения
    # Иначе time.perf_counter() покажет неправильное (слишком маленькое) время
    if device.type != "cpu":
        torch.cuda.synchronize() if device.type == "cuda" else torch.mps.synchronize()
    
    # Замер: 20 повторений и усреднение
    start = time.perf_counter()
    for _ in range(20):
        c = a @ b
    if device.type != "cpu":
        torch.cuda.synchronize() if device.type == "cuda" else torch.mps.synchronize()
    elapsed = (time.perf_counter() - start) / 20
    
    # Результат: чем меньше время, тем быстрее устройство
    print(f"Matrix multiply {n}x{n}: {elapsed*1000:.2f} ms")
    log.info("Бенчмарк %dx%d: %.2f мс", n, n, elapsed * 1000)


Matrix multiply 100x100: 0.47 ms
Бенчмарк 100x100: 0.47 мс
Matrix multiply 1000x1000: 0.79 ms
Бенчмарк 1000x1000: 0.79 мс


Matrix multiply 5000x5000: 63.74 ms
Бенчмарк 5000x5000: 63.74 мс


In [9]:
# === Итог ===
print("\n=== Environment check complete ===")
print(f"Summary: PyTorch {torch.__version__} on {device}")
log.info("Проверка окружения завершена — готовы к изучению трансформеров")



=== Environment check complete ===
Summary: PyTorch 2.8.0 on mps
Проверка окружения завершена — готовы к изучению трансформеров
